<h1 style="text-align:center">Data Preprocessing</h1>

## Imports

In [1]:
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import torchvision.transforms as transforms

## Reproducibility

In [2]:
SEED = 42

def seed_everything(seed=SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

## Load Dataset Manifest

In [3]:
df = pd.read_csv("../data/dataset_manifest.csv")
df.head()

,image_path,label
0,C:\Users\itxab\Downloads\AI\Brain_Tumor_Detect...,glioma
1,C:\Users\itxab\Downloads\AI\Brain_Tumor_Detect...,glioma
2,C:\Users\itxab\Downloads\AI\Brain_Tumor_Detect...,glioma
3,C:\Users\itxab\Downloads\AI\Brain_Tumor_Detect...,glioma
4,C:\Users\itxab\Downloads\AI\Brain_Tumor_Detect...,glioma


## Label Encoding (Fixed Mapping)

In [4]:
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

print(dict(zip(le.classes_, range(len(le.classes_)))))

{'glioma': 0, 'meningioma': 1, 'no_tumor': 2, 'pituitary': 3}


## Stratified Split (NO LEAKAGE)

In [5]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label_encoded"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_encoded"],
    random_state=SEED
)

print(f"Train: {len(train_df)}\nValidation: {len(val_df)}\nTest: {len(test_df)}")

Train: 3998
Validation: 857
Test: 857


## Class Weights

In [6]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["label_encoded"]),
    y=df["label_encoded"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print(class_weights)

tensor([1.0810, 1.0665, 0.8953, 0.9801])


## Clean Image Loader

In [7]:
def load_image(path):
    try:
        img = Image.open(path).convert("RGB")
        return img
    except:
        return None

## Transformation

### 1. Train

In [8]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ColorJitter(brightness=0.2, contrast=0.2),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### 2. Validation / Test

In [9]:
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Clean Dataset Class

In [10]:
class BrainTumorDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        path = self.df.loc[idx, "image_path"]
        label = self.df.loc[idx, "label_encoded"]

        image = load_image(path)

        if image is None:
            image = Image.new("RGB", (224, 224))

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

## Datasets

In [11]:
train_dataset = BrainTumorDataset(train_df, train_transform)
val_dataset = BrainTumorDataset(val_df, eval_transform)
test_dataset = BrainTumorDataset(test_df, eval_transform)

## Dataloaders

In [12]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

## Sanity Check

In [13]:
x, y = next(iter(train_loader))

print(x.shape)  
print(y.shape) 

torch.Size([16, 3, 224, 224])
torch.Size([16])
